In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev",
    choices=["fq_dev", "fq_test", "fq_prod"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="NETSUITE",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="cost",
    choices=["discount", "sales", "cost"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

# Get external location URLs
bronze_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_bronze`"
).select("url").collect()[0][0]

silver_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_silver`"
).select("url").collect()[0][0]

gold_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_gold`"
).select("url").collect()[0][0]

checkpoint = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_checkpoint`"
).select("url").collect()[0][0]

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_staging`"
).select("url").collect()[0][0]

print(f"Environment: {environment}")
print(f"Source: {source}")
print(f"Domain: {domain}")

In [0]:
%sql
select * from fq_dev_catalog.bronze.cost where effective_from_date = '2024-01-01' limit 1

In [0]:
from pyspark.sql.functions import *
import re, time

def to_snake_case(name):
    return re.sub(r'[\s\-]+', '_', name).lower()

def to_snake_case_df(df):
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, to_snake_case(col_name))
    return df

# Usage:
# cost = to_snake_case_df(cost)
# cost = enrich_json(cost)

def enrich_json(df_json):

    df_map = (
        spark.read.table('fq_dev_catalog.silver.dim_store')
        .select(
            col('deployment_name'),
            col('netsuite_location_name').alias('store_location')
        ).distinct()
    )

    df_enriched = (
        df_json.join(
            df_map,
            'store_location',
            'left'
        )
        # .drop(df_map.deployment_name)
        # .drop(df_json.deployment_name)
    )
    return df_enriched

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fq_dev_catalog.silver.cost (
    store_location STRING,
    upc_code STRING,
    item_name STRING,
    unit_cost DOUBLE,
    source_system STRING,
    deployment_name STRING,
    effective_from_date DATE,
    effective_to_date DATE,
    sys_id STRING
)
PARTITIONED BY ( -- Low cardinality columns first
    store_location
)
TBLPROPERTIES (
    delta.enableChangeDataFeed = true,
    delta.autoOptimize.optimizeWrite = true,
    delta.autoOptimize.autoCompact = true
);

In [0]:
def merge_stream_cost(df, i):
    try:
        exploded_df = (
            df.withColumn(
                "revision", explode(col("results.revisions"))
            )
            .withColumn(
                "location", explode(col("revision.assemblyLocations"))
            )
            .select(
                col("revision.AssemblyUpccode").alias("upc_code"),
                col("revision.AssemblyName").alias("item_name"),
                col("location.location").alias("store_location"),
                col("location.averageCost").alias("unit_cost"),
                col('effective_from_date'),
                col('effective_to_date')
            ).withColumn(
                "effective_from_date",
                to_date(col("effective_from_date"), "yyyyMMdd")
            ).withColumn(
                "effective_to_date",
                to_date(col("effective_to_date"), "yyyyMMdd")
            ).withColumn(
                "source_system", lit(f"{source}")
            ).withColumn(
                'sys_id', expr("uuid()")
            )
        )
        cost = to_snake_case_df(exploded_df)
        cost_upsert = enrich_json(cost)
        cost_upsert.createOrReplaceTempView("cost_upsert_microbatch")
       
        df.sparkSession.sql("""
            MERGE INTO fq_dev_catalog.silver.cost target
            USING (
                SELECT
                    *
                FROM cost_upsert_microbatch
            ) as source
            ON target.store_location = source.store_location
                AND target.upc_code = source.upc_code
                AND target.effective_from_date = source.effective_from_date
                AND target.effective_to_date = source.effective_to_date
            -- WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

        # df.sparkSession.sql("""
        # MERGE INTO fq_dev_catalog.silver.cost target
        # USING (
        #   SELECT store_location, upc_code, effective_from_date, effective_to_date, unit_cost, item_name
        #   FROM (
        #     SELECT *, ROW_NUMBER() OVER (
        #       PARTITION BY store_location, upc_code, effective_from_date, effective_to_date
        #       ORDER BY load_time DESC
        #     ) as rank
        #     FROM cost_upsert_microbatch
        #   )
        #   WHERE rank = 1
        # ) as source
        # ON target.store_id = source.store_id
        #    AND target.upc_code = source.upc_code
        #    AND target.effective_from_date = source.effective_from_date
        #    AND target.effective_to_date = source.effective_to_date
        # WHEN MATCHED THEN UPDATE SET *
        # WHEN NOT MATCHED THEN INSERT *
        # """)
    except Exception as e:
        print(f"Error in merge_stream: {e}")
        raise e

(spark.readStream
    # .option("schemaTrackingLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpointing_cost/schema_cost')
    .table("fq_dev_catalog.bronze.cost")
    .writeStream
    .foreachBatch(merge_stream_cost)
    .option("mergeSchema", "true")
    .option('skipChangeCommits', "true")
    .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_silver_cost3')  
    .trigger(availableNow=True)
    .start()
).awaitTermination()

time.sleep(20)

In [0]:
%sql
SELECT 
  count(distinct effective_from_date) as week_cardinality,
  COUNT(DISTINCT upc_code) AS upc_code_cardinality,
  COUNT(DISTINCT store_location) AS store_location_cardinality,
  COUNT(*) AS total_rows
FROM fq_dev_catalog.silver.cost;

In [0]:
%sql
OPTIMIZE fq_dev_catalog.silver.cost ZORDER BY (upc_code); -- high cardinality non-partitioned column first

In [0]:
%sql
select * from fq_dev_catalog.silver.cost limit 1 --where --deployment_name = 'ALBAIK - SQ DB03 - MALL OF EMIRATES - 1005003' and effective_from_date = '2026-01-02'

In [0]:
for query in spark.streams.active:
    query.stop()